# Multi-turn evaluators

This notebook shows how to evaluate **pre-recorded multi-turn conversations**
using the four Azure AI Foundry multi-turn evaluators:

| Evaluator                | Builtin name                       | Output     |
|--------------------------|------------------------------------|------------|
| Coherence                | `builtin.coherence`                | 1-5 score  |
| Task Completion          | `builtin.task_completion`          | Pass/Fail  |
| Customer Satisfaction    | `builtin.customer_satisfaction`    | 1-5 score  |
| Groundedness             | `builtin.groundedness`             | 1-5 score  |

The notebook will:

1. Load ~16 small conversation traces from `data/` (each labeled with an
   `expected_score`).
2. Convert the OpenAI-format messages into the `(query, response,
   tool_definitions)` shape Foundry's cloud evaluators expect.
3. Create a single evaluation object that bundles all four evaluators.
4. Submit one run with all traces as inline data, poll until it finishes.
5. Retrieve per-item results and report detailed stats: score
   distribution, agreement with the labeled `expected_score`, and a
   printout of failures.

> **Heads up:** Customer Satisfaction is the only one of the four whose
> sample script isn't published yet at the time of writing. We invoke it
> the same way as the other 1-5-scale evaluators; if your Foundry
> deployment hasn't enabled it, comment out its entry in the
> `EVALUATORS` list below.

## 1. Prerequisites

* Python 3.10+
* An Azure AI Foundry project with a deployed LLM (judge model).
* `az login` (the notebook uses `DefaultAzureCredential`).
* `pip install -r requirements.txt`

Set these environment variables (or put them in a `.env` next to the notebook):

* `FOUNDRY_PROJECT_ENDPOINT` — e.g. `https://<acct>.services.ai.azure.com/api/projects/<proj>`
* `FOUNDRY_MODEL_NAME` — judge model deployment name (e.g. `gpt-4o-mini`)

## 2. Imports and client setup

In [ ]:
import json
import os
import time
from pathlib import Path
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)

load_dotenv(override=True)

ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
# we recommend either GPT-5.x or a GPT-5.x-mini judge
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL_NAME")
print(f"Project endpoint: {ENDPOINT}")
print(f"Judge model deployment: {MODEL_DEPLOYMENT}")

Project endpoint: https://jcsantos-evaluator-api.services.ai.azure.com/api/projects/jcsantos-6777/
Judge model deployment: gpt-5.4-mini


## 3. Load sample traces from `data/`

In [68]:
DATA_DIR = Path("data")

traces = []
for path in sorted(DATA_DIR.glob("*.json")):
    with path.open(encoding="utf-8") as f:
        traces.append(json.load(f))

summary_df = pd.DataFrame(
    [
        {
            "test_id": t["test_id"],
            "evaluator": t["evaluator"],
            "expected_score": t["expected_score"],
            "num_messages": len(t["messages"]),
            "description": t["description"],
        }
        for t in traces
    ]
)
print(f"Loaded {len(traces)} traces from {DATA_DIR.resolve()}")
#widen the reason column for better readability in the notebook; adjust as needed for your display
pd.set_option("display.max_colwidth", 200)
summary_df

Loaded 15 traces from D:\src\foundry-samples\samples\python\multi-turn-evaluators\data


,test_id,evaluator,expected_score,num_messages,description
0,coherence_score1,coherence,"[1, 2]",6,"Agent completely ignores context, gives contradictory and unrelated responses"
1,coherence_score3,coherence,"[3, 4]",6,Agent mostly tracks context but has a noticeable irrelevant tangent and a weak transition
2,coherence_score5,coherence,"[4, 5]",6,"Perfect logical flow: agent tracks migration plan across turns, builds on each phase"
3,csat_score1,customer_satisfaction,"[1, 2]",6,"Agent dismisses user, provides no real help, user explicitly frustrated"
4,csat_score3,customer_satisfaction,"[2, 3]",6,Agent provides basic help but misses proactive solutions; customer left without full resolution
5,csat_score5,customer_satisfaction,5,8,"All requests handled perfectly, professional tone, complete resolution with proactive details"
6,csat_task_done_unsatisfied,customer_satisfaction,"[1, 2]",10,"Task technically completed but experience was poor — agent was curt, dismissive, and user expressed dissatisfaction"
7,ground_score1,groundedness,"[1, 2]",6,Agent completely ignores tool results and fabricates entirely different information
8,ground_score2,groundedness,"[1, 2]",4,"Agent fabricates specific facts not in tool result — invents a 60-day policy, system lookup capability, and free shipping that contradict the actual policy"
9,ground_score3,groundedness,3,4,Agent only asks clarifying questions and greets — zero factual propositions about the world


### Inspect one sample conversation

Each trace's `messages` list is plain OpenAI chat format
(`role`, `content`, optional `tool_calls`, `tool_call_id`). Some traces
include tool calls and tool results (for `task_completion`,
`groundedness`, and `customer_satisfaction`).

In [69]:
sample = next(t for t in traces if t["test_id"] == "tc_true")
print(f"Test id:        {sample['test_id']}")
print(f"Evaluator:      {sample['evaluator']}")
print(f"Expected score: {sample['expected_score']}")
print(f"Description:    {sample['description']}")
print()
for m in sample["messages"]:
    role = m.get("role")
    if m.get("tool_calls"):
        for tc in m["tool_calls"]:
            print(f"[{role} -> tool_call] {tc['function']['name']}({tc['function']['arguments']})")
    elif m.get("content"):
        content = m["content"]
        if isinstance(content, str) and len(content) > 200:
            content = content[:200] + "..."
        print(f"[{role}] {content}")

Test id:        tc_true
Evaluator:      task_completion
Expected score: TRUE
Description:    Agent fully completes a multi-turn task with refinements — flight booked and hotel found

[user] I need to book a flight from San Francisco to Tokyo for December 15th.
[assistant] I found several options for SFO to NRT on December 15th:

1. **ANA Flight NH7** — Direct, departs 11:30 AM, arrives 3:30 PM+1, $890
2. **United UA837** — Direct, departs 1:15 PM, arrives 5:10 PM+1, $9...
[user] Book the ANA flight. Also, can you book a hotel near Shinjuku station for 5 nights? Something under $150/night.
[assistant -> tool_call] book_flight({"flight": "NH7", "date": "2024-12-15", "passengers": 1})
[tool] Booking confirmed. Confirmation: ANA-NH7-DEC15-XK42M. E-ticket sent to your email.
[assistant -> tool_call] search_hotels({"location": "Shinjuku Station, Tokyo", "check_in": "2024-12-16", "nights": 5, "max_price": 150})
[tool] Found: 1) Hotel Gracery Shinjuku - $130/night, 0.2mi from station. 2) Citad

## 4. Convert traces to the Foundry conversation schema

Foundry's cloud evaluators expect each item to be split into three
fields:

* **`query`** — the *prompt* turns: system message (if any) + all `user`
  turns.
* **`response`** — the agent trajectory: all `assistant` turns (text and
  tool calls) plus the `tool` result messages.
* **`tool_definitions`** — OpenAI-style function schemas for the tools
  the agent had access to (optional but recommended for
  `task_completion` and `groundedness`).

Inside each message, the `content` is a list of typed parts:

* `{"type": "text", "text": "..."}`
* `{"type": "tool_call", "tool_call_id": "...", "name": "...", "arguments": {...}}`
* `{"type": "tool_result", "tool_result": ...}` (inside `role: "tool"` messages)

The helper below converts an OpenAI-format `messages` list into that
shape.

In [70]:
def _parse_args(args):
    if isinstance(args, str):
        try:
            return json.loads(args)
        except json.JSONDecodeError:
            return {"_raw": args}
    return args


def messages_to_foundry_qr(messages):
    """Split OpenAI-format messages into Foundry (query, response) arrays."""
    query = []
    response = []
    for msg in messages:
        role = msg.get("role")
        if role == "system":
            query.append({"role": "system", "content": [{"type": "text", "text": msg.get("content", "")}]})
        elif role == "user":
            query.append({"role": "user", "content": [{"type": "text", "text": msg.get("content", "")}]})
        elif role == "assistant":
            parts = []
            text = msg.get("content")
            if text:
                parts.append({"type": "text", "text": text})
            for tc in msg.get("tool_calls", []) or []:
                fn = tc.get("function", {})
                parts.append({
                    "type": "tool_call",
                    "tool_call_id": tc.get("id", ""),
                    "name": fn.get("name", ""),
                    "arguments": _parse_args(fn.get("arguments", "{}")),
                })
            if parts:
                response.append({"role": "assistant", "content": parts})
        elif role == "tool":
            content = msg.get("content", "")
            try:
                tool_result = json.loads(content) if isinstance(content, str) else content
            except (json.JSONDecodeError, TypeError):
                tool_result = content
            response.append({
                "role": "tool",
                "tool_call_id": msg.get("tool_call_id", ""),
                "content": [{"type": "tool_result", "tool_result": tool_result}],
            })
    return query, response


# Show the conversion on one sample
sample_q, sample_r = messages_to_foundry_qr(sample["messages"])
print("=== query ===")
print(json.dumps(sample_q, indent=2)[:600] + "...")
print()
print("=== response (first 2 entries) ===")
print(json.dumps(sample_r[:2], indent=2)[:800] + "...")

## 

=== query ===
[
  {
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "I need to book a flight from San Francisco to Tokyo for December 15th."
      }
    ]
  },
  {
    "role": "user",
    "content": [
      {
        "type": "text",
        "text": "Book the ANA flight. Also, can you book a hotel near Shinjuku station for 5 nights? Something under $150/night."
      }
    ]
  }
]...

=== response (first 2 entries) ===
[
  {
    "role": "assistant",
    "content": [
      {
        "type": "text",
        "text": "I found several options for SFO to NRT on December 15th:\n\n1. **ANA Flight NH7** \u2014 Direct, departs 11:30 AM, arrives 3:30 PM+1, $890\n2. **United UA837** \u2014 Direct, departs 1:15 PM, arrives 5:10 PM+1, $920\n3. **JAL JL1** \u2014 Direct, departs 5:00 PM, arrives 9:00 PM+1, $950\n\nWhich flight would you prefer, or should I filter by specific criteria?"
      }
    ]
  },
  {
    "role": "assistant",
    "content": [
      {
       

## 5. Build the inline items for the evaluation run

For each trace we emit one item containing `query`, `response`,
`tool_definitions` (empty here — none of the sample traces ship with
schemas) plus a few metadata fields (`test_id`, `expected_score`,
`evaluator`, `description`) so we can join the results back later.

We also save the built items to a JSONL file in Foundry format (`multi_turn_traces_foundry.jsonl`). If preferred, this file can be manually uploaded to Microsoft Foundry for evaluating inside the portal UI.
To upload from Foundry portal go to the project and then, on the left bar: Evaluations->Create->Target (Dataset)->Upload new dataset and then select the evaluators to run.

In [71]:
def trace_to_item(trace):
    q, r = messages_to_foundry_qr(trace["messages"])
    return {
        "test_id": trace["test_id"],
        "evaluator": trace["evaluator"],
        "expected_score": str(trace["expected_score"]),
        "description": trace["description"],
        "query": q,
        "response": r,
        "tool_definitions": [],
    }


items = [trace_to_item(t) for t in traces]
print(f"Built {len(items)} inline items.")
print("First item keys:", list(items[0].keys()))

# Save items as a JSONL file in Foundry format
output_jsonl_path = "multi_turn_traces_foundry.jsonl"
with open(output_jsonl_path, "w", encoding="utf-8") as f:
    for item in items:
        f.write(json.dumps(item) + "\n")
print(f"Saved {len(items)} items to {output_jsonl_path}")

Built 15 inline items.
First item keys: ['test_id', 'evaluator', 'expected_score', 'description', 'query', 'response', 'tool_definitions']
Saved 15 items to multi_turn_traces_foundry.jsonl


## 6. Create one evaluation per evaluator

Each trace in `data/` is labeled with the **specific** evaluator it's
designed to exercise. Running every trace through every evaluator would
produce noisy / off-target results (e.g. asking `groundedness` to score
a `coherence`-only trace).

So we group items by their target evaluator and create one
`evals.create(...)` per evaluator, each with a single
`TestingCriterionAzureAIEvaluator`. We'll submit one run per evaluator
in the next cell.


In [61]:
EVALUATORS = [
    ("coherence",             "builtin.coherence"),
    ("task_completion",       "builtin.task_completion"),
    ("customer_satisfaction", "builtin.customer_satisfaction"),
    ("groundedness",          "builtin.groundedness"),
]

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "test_id":        {"type": "string"},
            "evaluator":      {"type": "string"},
            "expected_score": {"type": "string"},
            "description":    {"type": "string"},
            "query":          {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
            "response":       {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
            "tool_definitions": {"anyOf": [{"type": "object"}, {"type": "array", "items": {"type": "object"}}]},
        },
        "required": ["query", "response"],
    },
    include_sample_schema=True,
)

# Group traces by their target evaluator.
items_by_evaluator: dict[str, list[dict]] = {}
for it in items:
    items_by_evaluator.setdefault(it["evaluator"], []).append(it)

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

# eval_objects[name] = (eval_id, builtin_name, matching_items)
eval_objects: dict[str, tuple[str, str, list[dict]]] = {}
for name, builtin in EVALUATORS:
    matching = items_by_evaluator.get(name, [])
    if not matching:
        print(f"[{name}] no matching traces, skipping")
        continue
    criterion = TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=name,
        evaluator_name=builtin,
        initialization_parameters={"deployment_name": MODEL_DEPLOYMENT},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "tool_definitions": "{{item.tool_definitions}}",
        },
    )
    eval_obj = openai_client.evals.create(
        name=f"Multi-turn evaluator: {name}",
        data_source_config=data_source_config,
        testing_criteria=[criterion],  # type: ignore[arg-type]
    )
    eval_objects[name] = (eval_obj.id, builtin, matching)
    print(f"[{name}] eval={eval_obj.id}  traces={len(matching)}")


[coherence] eval=eval_55c306055c6f48e9a33df9125b5c0fa2  traces=4
[task_completion] eval=eval_288fcc54f2de4b4d9deda31c14c16289  traces=3
[customer_satisfaction] eval=eval_6747e1a835d54cc4b9f21e63e08f9bd7  traces=4
[groundedness] eval=eval_98f36cf30cb84609b8e70696b5786925  traces=5


## 7. Submit one run per evaluator and poll until all finish

We submit each run, then poll them all in parallel (sleeping between
sweeps) until every run reaches a terminal state.


In [62]:
# Submit one run per evaluator (only the matching traces).
# eval_runs[name] = (eval_id, run_id)
eval_runs: dict[str, tuple[str, str]] = {}
for name, (eval_id, _builtin, matching) in eval_objects.items():
    run = openai_client.evals.runs.create(
        eval_id=eval_id,
        name=f"multi_turn_{name}_run",
        metadata={"sample": "multi-turn-evaluators",
                  "evaluator": name,
                  "num_items": str(len(matching))},
        data_source=CreateEvalJSONLRunDataSourceParam(
            type="jsonl",
            source=SourceFileContent(
                type="file_content",
                content=[SourceFileContentContent(item=it) for it in matching],
            ),
        ),
    )
    eval_runs[name] = (eval_id, run.id)
    print(f"[{name}] run={run.id} status={run.status}  ({len(matching)} traces)")

# Poll every run until all are terminal.
TERMINAL = ("completed", "failed", "canceled")
pending = dict(eval_runs)
final_runs: dict[str, object] = {}
start = time.time()
while pending:
    time.sleep(5)
    for name, (eid, rid) in list(pending.items()):
        run = openai_client.evals.runs.retrieve(eval_id=eid, run_id=rid)
        elapsed = time.time() - start
        print(f"  [{name}] status={run.status}  elapsed={elapsed:.0f}s")
        if run.status in TERMINAL:
            final_runs[name] = run
            del pending[name]

print()
for name, run in final_runs.items():
    print(f"[{name}] final={run.status}  counts={getattr(run, 'result_counts', None)}")
    print(f"   report: {getattr(run, 'report_url', None)}")


[coherence] run=evalrun_a0ab41a600d9478daa9479d4f20afceb status=in_progress  (4 traces)
[task_completion] run=evalrun_a228aafa1cf14a32884e2a111dcf6360 status=in_progress  (3 traces)
[customer_satisfaction] run=evalrun_07ac74e4266143d98110acf5be11019e status=in_progress  (4 traces)
[groundedness] run=evalrun_9e1a5dbad80a4354a4abd93a5c611920 status=in_progress  (5 traces)
  [coherence] status=completed  elapsed=6s
  [task_completion] status=completed  elapsed=6s
  [customer_satisfaction] status=completed  elapsed=7s
  [groundedness] status=in_progress  elapsed=7s
  [groundedness] status=in_progress  elapsed=12s
  [groundedness] status=in_progress  elapsed=18s
  [groundedness] status=completed  elapsed=24s

[coherence] final=completed  counts=ResultCounts(errored=0, failed=1, passed=3, total=4)
   report: https://ai.azure.com/nextgen/r/oZIOvVm3Txmvn16AWZ6I5A,babel-benchmark-rg,,jcsantos-evaluator-api,jcsantos-6777/build/evaluations/eval_55c306055c6f48e9a33df9125b5c0fa2/run/evalrun_a0ab41a

## 8. Retrieve and flatten per-item results

For each per-evaluator run we list its output items and flatten them
into a single long-form DataFrame keyed by `(test_id, evaluator)`.


In [63]:
def _to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if isinstance(obj, dict):
        return obj
    return dict(obj)


rows = []
total_output_items = 0
for name, (eid, rid) in eval_runs.items():
    run_output_items = list(
        openai_client.evals.runs.output_items.list(eval_id=eid, run_id=rid)
    )
    total_output_items += len(run_output_items)
    print(f"[{name}] retrieved {len(run_output_items)} output items")
    for oi in run_output_items:
        oi_d = _to_dict(oi)
        sample = oi_d.get("sample") or {}
        sample_item = (sample.get("input") or {}) if isinstance(sample, dict) else {}
        # Different SDK versions surface the original item under
        # slightly different keys; try a few.
        item = (
            oi_d.get("datasource_item")
            or oi_d.get("data_source_item")
            or sample_item
            or {}
        )
        test_id = item.get("test_id", oi_d.get("id", "?"))
        expected = item.get("expected_score")
        for res in oi_d.get("results", []) or []:
            rows.append({
                "test_id": test_id,
                "evaluator": res.get("name") or res.get("metric") or name,
                "expected_score": expected,
                "score": res.get("score"),
                "label": res.get("label"),
                "passed": res.get("passed"),
                "threshold": res.get("threshold"),
                "reason": (res.get("reason") or "")[:200],
            })

print(f"\nTotal output items across runs: {total_output_items}")
results_df = pd.DataFrame(rows)
results_df.head(20)

[coherence] retrieved 4 output items
[task_completion] retrieved 3 output items
[customer_satisfaction] retrieved 4 output items
[groundedness] retrieved 5 output items

Total output items across runs: 16


,test_id,evaluator,expected_score,score,label,passed,threshold,reason
0,coherence_score1,coherence,"[1, 2]",2.0,fail,False,3,"The response contains abrupt topic changes and irrelevant answers, so it is difficult to follow and does not address the user’s request coherently."
1,coherence_score3,coherence,"[3, 4]",3.0,pass,True,3,"The answer is mostly organized and relevant, but one off-topic insertion interrupts the logical flow."
2,coherence_score5,coherence,5,4.0,pass,True,3,"The response is well organized, directly relevant, and maintains a clear logical flow across the migration phases."
3,coherence_skip,coherence,None,3.0,pass,True,3,"The response is readable and structured, but it does not logically track the different user requests, so the flow is only partially coherent."
4,tc_false_evolving,task_completion,FALSE,0.0,fail,False,1,"The user requested a class with methods for Celsius-Fahrenheit, Fahrenheit-Celsius, and Kelvin conversions. The agent only provided standalone functions for Celsius/Fahrenheit in both directions a..."
5,tc_false_incomplete,task_completion,FALSE,0.0,fail,False,1,"The agent summarized the sales data, but the user explicitly requested visualizations and asked where the charts were. No bar chart or line chart was provided, so the deliverable is incomplete."
6,tc_true,task_completion,TRUE,1.0,pass,True,1,"The agent successfully booked the ANA flight and a hotel near Shinjuku Station for 5 nights under $150/night, with confirmations provided. The final outcome is actionable and matches the user's re..."
7,csat_score1,customer_satisfaction,"[1, 2]",2.0,fail,False,3,"The agent responded politely, but it did not meet the customer's stated need for a replacement or refund after a missing $200 package. It only repeated the delivery status and pushed the issue to ..."
8,csat_score3,customer_satisfaction,"[2, 3]",2.0,fail,False,3,"The agent gave a few generic troubleshooting steps, but it didn’t address the core issue that the account is still locked after two hours. It also gave inconsistent timing information and never pr..."
9,csat_score5,customer_satisfaction,5,5.0,pass,True,3,"The agent fully handled the cancellation, confirmed a prorated refund, and clearly answered the data-retention question. It provided all key details efficiently, so the customer would likely feel ..."


## 9. Stats

### 9a. Score distribution per evaluator

In [64]:
def _norm(v):
    if v is None:
        return None
    s = str(v).strip().lower()
    if s in ("true", "pass", "passed"):
        return "pass"
    if s in ("false", "fail", "failed"):
        return "fail"
    return s


dist = (
    results_df.assign(score_norm=lambda d: d["score"].fillna(d["label"]).map(_norm))
    .groupby(["evaluator", "score_norm"])
    .size()
    .unstack(fill_value=0)
)
dist

score_norm,0.0,1.0,2.0,3.0,4.0,5.0
evaluator,,,,,,
coherence,0,0,1,2,1,0
customer_satisfaction,0,0,3,0,0,1
groundedness,0,0,2,1,1,1
task_completion,2,1,0,0,0,0


### 9b. Agreement with `expected_score`

In [65]:
def parse_to_set(v, is_bool=False):
    """Normalize a scalar, list, or stringified list into a set of canonical values."""
    if v is None or pd.isna(v):
        return set()
    if isinstance(v, str) and v.strip().startswith("[") and v.strip().endswith("]"):
        try:
            v = json.loads(v)
        except json.JSONDecodeError:
            pass
    items = v if isinstance(v, list) else [v]
    parsed = []
    for x in items:
        if x is None or pd.isna(x):
            continue
        s = str(x).strip().lower()
        if is_bool and s in ("true", "pass", "passed", "1", "1.0", "false", "fail", "failed", "0", "0.0"):
            parsed.append("pass" if s in ("true", "pass", "passed", "1", "1.0") else "fail")
        else:
            try:
                parsed.append(float(s))
            except ValueError:
                parsed.append(s)
    return set(parsed)


# Pick a usable actual value from available fields
actual_col = results_df["score"].fillna(results_df["label"]).fillna(results_df["passed"])

exact_list, off_by_one_list = [], []
for _, row in results_df.assign(actual=actual_col).iterrows():
    is_tc = row["evaluator"] == "task_completion"
    a_set = parse_to_set(row["actual"], is_tc)
    e_set = parse_to_set(row["expected_score"], is_tc)
    
    exact = bool(a_set & e_set)
    exact_list.append(exact)
    off_by_one_list.append(
        not exact and any(abs(x - y) <= 1 for x in a_set for y in e_set if isinstance(x, float) and isinstance(y, float))
    )

agree = results_df.assign(
    actual=actual_col,
    exact=exact_list,
    off_by_one=off_by_one_list,
)

# Aggregate results per evaluator
agreement_summary = (
    agree.groupby("evaluator")
    .agg(n=("test_id", "count"),
         exact_match=("exact", "sum"),
         off_by_one=("off_by_one", "sum"))
    .assign(exact_rate=lambda d: (d["exact_match"] / d["n"]).round(2),
            within_one_rate=lambda d: ((d["exact_match"] + d["off_by_one"]) / d["n"]).round(2))
)
agreement_summary


,n,exact_match,off_by_one,exact_rate,within_one_rate
evaluator,,,,,
coherence,4,2,1,0.5,0.75
customer_satisfaction,4,4,0,1.0,1.00
groundedness,5,5,0,1.0,1.00
task_completion,3,3,0,1.0,1.00


### 9c. Failures (actual score didn't match expected)

In [66]:
fails = agree[~agree["exact"]].copy()
print(f"{len(fails)} mismatch(es) out of {len(agree)} judgments\n")
for _, row in fails.iterrows():
    print(f"- {row['test_id']:<32} {row['evaluator']:<22} "
          f"expected={row['expected_score']!s:<6} actual={row['actual']!s:<6}")
    if row.get("reason"):
        print(f"    reason: {row['reason']}")

2 mismatch(es) out of 16 judgments

- coherence_score5                 coherence              expected=5      actual=4.0   
    reason: The response is well organized, directly relevant, and maintains a clear logical flow across the migration phases.
- coherence_skip                   coherence              expected=None   actual=3.0   
    reason: The response is readable and structured, but it does not logically track the different user requests, so the flow is only partially coherent.


## 10. Cleanup (optional)

Delete the evaluation object to keep your project tidy. Uncomment to
run.

In [49]:
for name, (eval_id, _builtin, _matching) in eval_objects.items():
    openai_client.evals.delete(eval_id=eval_id)
    print(f"Deleted [{name}] eval {eval_id}")


Deleted [coherence] eval eval_5c78ba087ad24686af718903bef0a7d5
Deleted [task_completion] eval eval_cd5be70a57644c6b93a1999b8b79e780
Deleted [customer_satisfaction] eval eval_547b2c0aece94e5987841c0ddb0315ac
Deleted [groundedness] eval eval_61bd2012d3124bc1b1ca702f071aec47


## Next steps (optional)

* Add your own traces under `data/` (same JSON shape) and rerun.
* Add a `tool_definitions` field to traces so `task_completion` /
  `groundedness` can validate that the agent invoked the right tools
  with the right arguments.
* Run the same eval object multiple times to measure judge reliability:
  call `openai_client.evals.runs.create(...)` repeatedly with the same
  `eval_id` and aggregate the per-run results.
* Browse the full reports for each run from the **Report URL** printed
  in section 7, or in your Foundry project's Evaluations tab.